<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow: HLLC Component — Modeling Flow Over Complex Terrain

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the usage of the `RiverFlowDynamics_HLLC` Landlab component with a real-world digital elevation model (DEM). The component applies an **HLLC Godunov-type finite-volume scheme** for the depth-averaged 2D shallow-water equations on a RasterModelGrid.

> **Key numerical differences from `RiverFlowDynamics` (Casulli & Cheng 1992):**
>
> | Feature | RiverFlowDynamics | RiverFlowDynamics_HLLC |
> |---|---|---|
> | Scheme | Semi-implicit, semi-Lagrangian | Explicit Godunov-type (HLLC flux) |
> | Time stepping | User-supplied fixed dt (can exceed CFL) | Adaptive CFL (or user-supplied with warning) |
> | Shock capturing | No | Yes — handles hydraulic jumps and bores |
> | Wet/dry fronts | Via threshold depth | Positive-depth guarantee + hydrostatic reconstruction |
> | Velocity storage | Scalar at links | x- and y-components at nodes |
> | Required input fields | depth, elevation, velocity (link) | depth, topographic elevation only |
>

### Theory

The HLLC solver integrates the shallow-water equations in conservation form. Fluxes are computed with the HLLC approximate Riemann solver; topographic source terms use Audusse hydrostatic reconstruction for exact well-balancedness on wet beds. Time integration uses Strang operator splitting alternating x- and y-sweeps.

### Import the needed libraries:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from IPython.display import clear_output
import time

from landlab import RasterModelGrid
from landlab.components import RiverFlowDynamics_HLLC
from landlab.io import esri_ascii
from landlab.plot.imshow import imshow_grid

## Information about the component

In [ ]:
help(RiverFlowDynamics_HLLC)

-- --
## Example 2: Modeling water flow over complex terrain (Kootenai River side-channel)

In this example, we import a digital elevation model (DEM) for a side-channel of the Kootenai River, Idaho, US. We demonstrate how `RiverFlowDynamics_HLLC` simulates water flow over real-world terrain.

### Loading the Digital Elevation Model

We load the DEM from an ASCII file and use it as our topographic elevation field.

In [ ]:
# Getting the grid and some parameters
asc_file = "DEM_kootenai_37x50_1x1.asc"
with open(asc_file) as fp:
    grid = esri_ascii.load(fp, name="topographic__elevation")
te = grid.at_node["topographic__elevation"]

### Define Simulation Parameters

We specify Manning's roughness coefficient and simulation duration.

In [ ]:
# Basic parameters
mannings_n = 0.035      # Manning's roughness coefficient, [s/m^(1/3)]

# Simulation parameters
target_time            = 120.0  # Total simulated time [s]
display_animation_freq = 10     # Redraw every this many simulated seconds

### Visualize Initial Topography

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
te = grid.at_node["topographic__elevation"]
im = ax.imshow(
    te.reshape(grid.shape),
    origin="lower",
    extent=[
        grid.x_of_node.min(), grid.x_of_node.max(),
        grid.y_of_node.min(), grid.y_of_node.max(),
    ],
    cmap="terrain",
    vmin=538,
    vmax=543,
)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.20, label="Elevation (m)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Kootenai River Side-Channel Topography", fontsize=14)
plt.tight_layout()
plt.show()

### Set Initial Conditions

With `RiverFlowDynamics_HLLC`, only the **depth** field is required. The component automatically creates `surface_water__elevation`, `surface_water__x_velocity`, `surface_water__y_velocity`, and momentum fields.

In [ ]:
# Initial conditions: empty channel — only depth field required
h = grid.add_zeros("surface_water__depth", at="node")

### Set Boundary Conditions

Water flows from **right to left** (negative x-direction). In `RiverFlowDynamics_HLLC`, boundary velocities are specified as **x- and y-components at nodes**, not as scalar link velocities. Negative `u_values` means leftward flow.

In [ ]:
# Entry boundary nodes at the right edge
fixed_entry_nodes = grid.nodes_at_right_edge

entry_nodes_h_values = np.array([
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.20,
    0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20,
    0.20, 0.20, 0.20, 0.20, 0.00, 0.00, 0.0, 0.0, 0.0,])

u_magnitude = 0.50  # x-velocity magnitude [m/s], negative = leftward flow

entry_nodes_u_values = np.where(entry_nodes_h_values > 0.0, -u_magnitude, 0.0)
entry_nodes_v_values = np.zeros(len(fixed_entry_nodes))

### Visualize Entry Cross-section

In [ ]:
# Extract coordinates for the right boundary
y_ground = grid.y_of_node[grid.nodes_at_right_edge]
z_ground = te[grid.nodes_at_right_edge]

constant_wse = np.mean(entry_nodes_h_values + te[fixed_entry_nodes])

fig, ax = plt.subplots(figsize=(5, 4))

base_elevation = z_ground.min() - 0.2
ax.fill_between(y_ground, base_elevation, z_ground, color='saddlebrown', alpha=0.4, zorder=1)
ax.plot(y_ground, z_ground, color='saddlebrown', linewidth=2.5, zorder=3)

wse_array = np.full_like(z_ground, constant_wse)
ax.fill_between(y_ground, z_ground, wse_array, where=(z_ground <= wse_array),
                color='deepskyblue', alpha=0.5, interpolate=True, zorder=2)

for i in range(len(y_ground) - 1):
    z1, z2 = z_ground[i], z_ground[i+1]
    y1, y2 = y_ground[i], y_ground[i+1]
    
    if z1 <= constant_wse and z2 <= constant_wse:
        ax.plot([y1, y2], [constant_wse, constant_wse], color='dodgerblue', linewidth=2.5, zorder=4)
    elif (z1 - constant_wse) * (z2 - constant_wse) < 0:
        slope_b = (z2 - z1) / (y2 - y1)
        y_int = y1 + (constant_wse - z1) / slope_b
        if z1 <= constant_wse:
            ax.plot([y1, y_int], [constant_wse, constant_wse], color='dodgerblue', linewidth=2.5, zorder=4)
        else:
            ax.plot([y_int, y2], [constant_wse, constant_wse], color='dodgerblue', linewidth=2.5, zorder=4)

ax.set_title("Entry Cross-section at Right Boundary", fontsize=14, fontweight='bold')
ax.set_xlabel("Distance along cross-section [m]", fontsize=12)
ax.set_ylabel("Elevation [m]", fontsize=12)
ax.set_xlim(y_ground.min(), y_ground.max())
ax.set_ylim(base_elevation, z_ground.max() + 0.1)
ax.grid(True, linestyle='--', alpha=0.6, zorder=0)

water_line = Line2D([0], [0], color='dodgerblue', lw=2.5, label='Water Surface')
bed_line   = Line2D([0], [0], color='saddlebrown', lw=2.5, label='Channel Bed')
ax.legend(handles=[bed_line, water_line], loc='upper right', framealpha=1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Initialize the RiverFlowDynamics_HLLC Component

Key differences from `RiverFlowDynamics`:
- **No `dt` argument** at construction — adaptive CFL stepping by default.
- **`entry_nodes_u_values` / `entry_nodes_v_values`** replace `entry_links_vel_values`.
- **No `fixed_entry_links`** — the HLLC scheme is node-based.
- All grid edges use **transmissive (zero-gradient) outflow** by default; add `wall_edges` for closed walls.

In [ ]:
# Initialize the HLLC component
rfd = RiverFlowDynamics_HLLC(
    grid,
    mannings_n=mannings_n,
    fixed_entry_nodes=fixed_entry_nodes,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_nodes_u_values=entry_nodes_u_values,
    entry_nodes_v_values=entry_nodes_v_values,
    # No wall_edges: all domain boundaries are transmissive (open outflow)
)

### Run the Simulation

We run for 120 seconds. With CFL-based stepping, the simulated time per step varies depending on wave speeds.

In [ ]:
t0             = time.time()
next_display_t = 0.0
nrows, ncols   = grid.shape
X = grid.x_of_node.reshape((nrows, ncols))
Y = grid.y_of_node.reshape((nrows, ncols))
Z = grid.at_node["topographic__elevation"].reshape((nrows, ncols))

while rfd.elapsed_time < target_time:
    rfd.run_one_step()

    if rfd.elapsed_time >= next_display_t:
        clear_output(wait=True)

        elapsed_wall = time.time() - t0
        pct = rfd.elapsed_time / target_time
        eta = elapsed_wall * (1.0 / pct - 1.0) if pct > 0 else float("nan")
        print(f"sim-time {rfd.elapsed_time:.2f} / {target_time:.1f} s  "
              f"({pct:.1%})  wall {elapsed_wall:.1f} s  ETA {eta:.1f} s  "
              f"dt={rfd.current_dt:.4f} s")

        h_arr      = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
        wse        = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))
        wse_masked = np.ma.masked_where(h_arr <= 0.001, wse)

        fig, ax = plt.subplots(figsize=(6, 6))
        bed = ax.pcolormesh(X, Y, Z, cmap="gray", shading="auto", vmin=538, vmax=543)
        wat = ax.pcolormesh(X, Y, wse_masked, cmap="Blues", shading="auto",
                            alpha=0.8, vmin=538, vmax=540)

        divider = make_axes_locatable(ax)
        cax1 = divider.append_axes("right", size="5%", pad=0.15)
        cax2 = divider.append_axes("right", size="5%", pad=0.85)
        fig.colorbar(wat, cax=cax1, label="Water Surface Elevation (m)")
        fig.colorbar(bed, cax=cax2, label="Bed Elevation (m)")

        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(f"Hydrodynamic Routing  [t = {rfd.elapsed_time:.2f} s]",
                     fontsize=11)
        plt.tight_layout()
        plt.show()

        next_display_t += display_animation_freq

### Visualize Final Results

Let's examine the water depth and water surface elevation at the end of the simulation.

In [ ]:
H = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
H_masked = np.ma.masked_where(H <= 0.001, H)

fig, ax = plt.subplots(figsize=(6, 6))
bed = ax.pcolormesh(X, Y, Z, cmap="gray", shading="auto", vmin=538, vmax=543)
wat = ax.pcolormesh(X, Y, H_masked, cmap="Blues", shading="auto", alpha=0.8)

divider = make_axes_locatable(ax)
cax1 = divider.append_axes("right", size="5%", pad=0.05)
cax2 = divider.append_axes("right", size="5%", pad=0.55)

fig.colorbar(wat, cax=cax1, label="Water Depth (m)")
fig.colorbar(bed, cax=cax2, label="Bed Elevation (m)")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title(f"Final Water Depth at t = {rfd.elapsed_time:.2f} s",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
h_arr      = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
wse        = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))
wse_masked = np.ma.masked_where(h_arr <= 0.001, wse)

fig, ax = plt.subplots(figsize=(6, 6))
bed = ax.pcolormesh(X, Y, Z, cmap="gray", shading="auto", vmin=538, vmax=543)
wat = ax.pcolormesh(X, Y, wse_masked, cmap="Blues", shading="auto", alpha=0.8)

divider = make_axes_locatable(ax)
cax1 = divider.append_axes("right", size="5%", pad=0.05)
cax2 = divider.append_axes("right", size="5%", pad=0.55)
fig.colorbar(wat, cax=cax1, label="Water Surface Elevation (m)")
fig.colorbar(bed, cax=cax2, label="Bed Elevation (m)")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title(f"Final Water Surface Elevation at t = {rfd.elapsed_time:.2f} s",
             fontsize=11)
plt.tight_layout()
plt.show()

### Visualize Flow Velocity Magnitude

With `RiverFlowDynamics_HLLC`, velocities are stored as **x- and y-components at nodes**. Velocity magnitude is computed directly as $|\mathbf{u}| = \sqrt{u^2 + v^2}$.

In [ ]:
# Compute velocity magnitude from node-centered velocity components
u = grid.at_node["surface_water__x_velocity"].reshape((nrows, ncols))
v = grid.at_node["surface_water__y_velocity"].reshape((nrows, ncols))
vel_mag    = np.sqrt(u**2 + v**2)
h_arr      = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
vel_masked = np.ma.masked_where(h_arr <= 0.001, vel_mag)

fig, ax = plt.subplots(figsize=(6, 6))
bed = ax.pcolormesh(X, Y, Z, cmap="gray", shading="auto", vmin=538, vmax=543)
vel = ax.pcolormesh(X, Y, vel_masked, cmap="plasma", shading="auto", alpha=0.8)

divider = make_axes_locatable(ax)
cax1 = divider.append_axes("right", size="5%", pad=0.05)
cax2 = divider.append_axes("right", size="5%", pad=0.55)
fig.colorbar(vel, cax=cax1, label="Velocity Magnitude (m/s)")
fig.colorbar(bed, cax=cax2, label="Bed Elevation (m)")

ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title(f"Flow Velocity Magnitude at t = {rfd.elapsed_time:.2f} s",
             fontsize=11)
plt.tight_layout()
plt.show()

### Analyze Cross-sections

In [ ]:
# Extract grid dimensions and middle row
nrows, ncols = grid.shape
mid_row = nrows // 2

x_profile   = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
z_profile   = grid.at_node['topographic__elevation'].reshape((nrows, ncols))[mid_row, :]
wse_profile = grid.at_node['surface_water__elevation'].reshape((nrows, ncols))[mid_row, :]
depth_profile = grid.at_node['surface_water__depth'].reshape((nrows, ncols))[mid_row, :]

fig, ax = plt.subplots(figsize=(5, 4))

base_elevation = z_profile.min() - 0.5
ax.fill_between(x_profile, base_elevation, z_profile, color='saddlebrown', alpha=0.4, zorder=1)
ax.plot(x_profile, z_profile, color='saddlebrown', linewidth=2.5, label='Channel Bed', zorder=3)

wse_line = np.where(depth_profile > 0.001, wse_profile, np.nan)
ax.plot(x_profile, wse_line, color='dodgerblue', linewidth=2.5, label='Water Surface', zorder=4)
ax.fill_between(x_profile, z_profile, wse_profile,
                where=(wse_profile > z_profile + 0.001),
                color='deepskyblue', alpha=0.5, interpolate=True, zorder=2)

ax.set_title('Longitudinal Profile (East-West) through Domain Center', fontsize=14, fontweight='bold')
ax.set_xlabel('Distance Along Channel [m]', fontsize=12)
ax.set_ylabel('Elevation [m]', fontsize=12)
ax.set_xlim(x_profile.min(), x_profile.max())

max_water_elev = np.nanmax(wse_line) if np.any(depth_profile > 0.001) else z_profile.max()
ax.set_ylim(base_elevation, max(max_water_elev, z_profile.max()) + 0.2)

ax.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax.legend(loc='upper right', framealpha=1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Interpretation of Results

The simulation demonstrates `RiverFlowDynamics_HLLC` routing water over real Kootenai River DEM terrain. Key observations:

1. **Channel-following flow**: Water naturally follows topographic lows, consistent with real-world hydraulics.
2. **HLLC advantages on real DEMs**: The hydrostatic reconstruction ensures exact well-balancedness on wet terrain — no spurious velocities arise from topographic gradients alone. This is especially important over complex bathymetry.
3. **Larger inundation extent**: The HLLC shock-capturing scheme correctly propagates the wet/dry front, capturing more inundation than the semi-implicit RFD solver which tends to under-predict frontal speeds.
4. **Velocity from node fields**: Unlike `RiverFlowDynamics`, velocity magnitude is computed directly from the conserved momenta ($hu$, $hv$) stored as node fields — no interpolation to links needed.

-- --
### And that's it!

You now know how to use the `RiverFlowDynamics_HLLC` component to simulate water flow over real-world DEMs :)

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>